# Setup del cuaderno

In [2]:
### download modules
%pip install pandas numpy

### load modules
import os 
import sys
import pandas as pd
import numpy as np

## 1. Descriptivas de la base

### 1.1 Importar datos

In [5]:
### Cargar datos seleccionando las columnas requeridas
db = pd.read_csv("../stores/input/01_original_data_train.csv",
                 usecols=["pobre", "urbano", "numero_personas_hogar",
                          "cantidad_cuartos","arriendo","regimen_salud_2_household_average",
                          "edad_menor_18_household_average","oc_household_average",
                          "maximo_nivel_educativo_1_household_average",
                          "hh_female","hh_informal","hh_oc"])

### Transformar la variable dependiente
db["pobre"] = np.where(db["pobre"] == 'Yes', 1, 0)

### Computar informalidad condicional a estar trabajando y eliminar variable ocupado
db.loc[db['hh_oc'] == 0, 'hh_informal'] = np.nan
db = db.drop(columns=['hh_oc'])

### 1.2 Preparar tabla

In [7]:
### Calcular tamaños de muestra (N) para los encabezados
n_pobre = db[db['pobre'] == 1].shape[0]
n_nopobre = db[db['pobre'] == 0].shape[0]

col_pobre = f"Pobre\n(N={n_pobre:,}, {n_pobre/(n_pobre+n_nopobre)*100:.2f}%)"
col_nopobre = f"No Pobre\n(N={n_nopobre:,}, {n_nopobre/(n_pobre+n_nopobre)*100:.2f}%))"

In [8]:
rows = []

### Variables continuas: formato "Promedio (Desv Est)"
continuas = {
    'numero_personas_hogar': 'Personas por hogar',
    'cantidad_cuartos': 'Cantidad de cuartos',
    "arriendo": "Arriendo (COP)"
}

for col, nombre in continuas.items():
    mean_p = db.loc[db['pobre'] == 1, col].mean()
    std_p = db.loc[db['pobre'] == 1, col].std()
    
    mean_np = db.loc[db['pobre'] == 0, col].mean()
    std_np = db.loc[db['pobre'] == 0, col].std()
    
    # Ajuste para Arriendo (COP)
    if col == "arriendo":
        rows.append({
            'Variable': nombre,
            col_pobre: f"{mean_p:,.0f} ({std_p:,.0f})",
            col_nopobre: f"{mean_np:,.0f} ({std_np:,.0f})"
        })
    else:
        rows.append({
            'Variable': nombre,
            col_pobre: f"{mean_p:.2f} ({std_p:.2f})",
            col_nopobre: f"{mean_np:.2f} ({std_np:.2f})"
        })

# Variables categóricas/binarias: formato "XX.XX%"
categoricas = {
    'urbano': '¿El hgogar está ubicado zona urbana?',
    "regimen_salud_2_household_average":"Miembros del hogar en regimen subsidiado",
    "edad_menor_18_household_average":"Miembros del hogar menores a 18 años",
    "oc_household_average":"Miembros del hogar ocupados",
    "maximo_nivel_educativo_1_household_average": "Miembros del hogar educación primaria máximo",
    "hh_female": "¿Jefe del hogar es mujer? (%)",
    "hh_informal": "¿Jefe del hogar es un trabajador informal?(%)"
}

for col, nombre in categoricas.items():
    prop_p = db.loc[db['pobre'] == 1, col].mean() * 100
    prop_np = db.loc[db['pobre'] == 0, col].mean() * 100
    
    rows.append({
        'Variable': nombre,
        col_pobre: f"{prop_p:.2f}%",
        col_nopobre: f"{prop_np:.2f}%"
    })

In [9]:
# Construir y mostrar el DataFrame final
tabla_descriptiva = pd.DataFrame(rows)
print(tabla_descriptiva)

                                        Variable Pobre\n(N=19,195, 20.72%)  \
0                             Personas por hogar               4.16 (2.05)   
1                            Cantidad de cuartos               3.05 (1.12)   
2                                 Arriendo (COP)       274,089 (2,742,312)   
3           ¿El hgogar está ubicado zona urbana?                    85.40%   
4       Miembros del hogar en regimen subsidiado                     0.77%   
5           Miembros del hogar menores a 18 años                    37.01%   
6                    Miembros del hogar ocupados                    31.31%   
7   Miembros del hogar educación primaria máximo                     9.74%   
8                  ¿Jefe del hogar es mujer? (%)                    46.79%   
9  ¿Jefe del hogar es un trabajador informal?(%)                     8.92%   

  No Pobre\n(N=73,440, 79.28%))  
0                   3.09 (1.65)  
1                   3.49 (1.26)  
2           499,678 (3,710,786)  
3    

### 1.3 Exportar

In [10]:
tabla_descriptiva.to_latex('../stores/output/01_descriptives_main_document.tex',float_format="{:.2f}".format,escape=True,index=False)